| Step | Purpose |
|---|---|
| 5.1 | Trigger phrases → a 120-character window after each |
| 5.2 | County, state and year inside the window |
| 5.3 | "Aforesaid": borrow county/state from the page header |
| 5.4 | Match to `places` (county first, then state) |
| 5.5 | Clean up: dedupe, one current residence per person, order the chain (`seq`) |
| 5.6 | Fallback: most-frequent state for files with no current residence |
| 5.7 | Assertions + save `interim/residences` |

In [1]:
import sys
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
import config
from rx import io, text, places, remote

In [2]:
import pandas as pd
d = io.read_stage("interim", "decl_text")
pl = io.read_stage("processed", "places")

STATES = ("massachusetts|connecticut|new york|new jersey|pennsylvania|virginia|north carolina|south carolina|"
          "new hampshire|vermont|maryland|delaware|georgia|rhode island|maine|kentucky|tennessee|ohio|indiana|"
          "illinois|missouri|alabama|mississippi|michigan|louisiana|arkansas|wisconsin|iowa|florida|district of columbia")
ABBR = r"mass|conn|vt|penn|tenn|ky|n\. ?[yhcj]\.|s\. ?c\.|va\.|md\.|ga\.|del\.|r\. ?i\."
STATE_RX = rf"\b({STATES}|{ABBR})(?![a-z])"
TRIG = {
    "now_resides": r"(?:a |now )?(?:resident|inhabitant)s? (?:of|in)|now (?:resides?|residing|lives?|living|dwells?) (?:in|at)|residing (?:in|at)",
    "born":        r"\bborn (?:in|at)",
    "enlisted":    r"(?:living|lived|resided|residing) (?:in|at)(?=.{0,100}when (?:he|i|called|the war))",
    "removed_to":  r"(?:removed|moved|emigrated) (?:with (?:his|her|my) family )?(?:to|into)(?! the (?:army|service|south|north))",
    "lived_since": r"(?:has|have|hath) (?:resided|lived|remained) (?:in|at)(?=.{0,100}ever since)",
    "married":     r"\bmarried (?:to )?(?:.{0,40}? )?(?:in|at)(?= )",
}

# 5.1
hits = []
for kind, trig in TRIG.items():
    m = d.text.str.extractall(rf"(?:{trig})(.{{0,120}})")[0].rename("window").reset_index()
    if kind == "now_resides":
        m = m.drop_duplicates("level_0")
    hits.append(m.assign(kind=kind))
r = pd.concat(hits, ignore_index=True).rename(columns={"level_0": "row"})
r = r.join(d[["NAID", "pageNumber", "person", "app_year"]], on="row")
print(r.kind.value_counts())

# 5.2
w = r.window
r["county_raw"] = w.str.extract(r"(?:county|parish) of ([a-z' .]{2,25}?)(?=[,;.]| and | in | state| commonwealth|$)")[0]
r["county_raw"] = r.county_raw.fillna(w.str.extract(r"(?:^|[ ,])([a-z']+(?: [a-z']+)?) (?:county|co\.)")[0])
r["state_raw"] = w.str.extract(STATE_RX)[0]
r["year"] = pd.to_numeric(w.str.extract(r"\b(1[78]\d\d)\b")[0], errors="coerce").astype("Int64")

# 5.3
head = d.text.str[:300]
d["hdr_state"] = head.str.extract(STATE_RX)[0]
d["hdr_county"] = head.str.extract(r"(?:county|parish) of ([a-z' .]{2,25}?)(?=[,;.:}]| and | ss|$)")[0].fillna(
                  head.str.extract(r"(?:^|[ ,}])([a-z']+(?: [a-z']+)?) (?:county|co\.)")[0])
r = r.join(d[["hdr_state", "hdr_county"]], on="row")
said = w.str.contains(r"^.{0,40}(?:aforesaid|said county|said state|same county|this county)")
use_hc = said & r.county_raw.isna()
r.loc[use_hc, "county_raw"] = r.loc[use_hc, "hdr_county"]
use_hs = r.state_raw.isna() & (said | (r.kind == "now_resides"))
r.loc[use_hs, "state_raw"] = r.loc[use_hs, "hdr_state"]
r["state"] = places.normalise_state_series(r.state_raw)

# 5.4
r["county_key"] = (r.county_raw.str.replace(r"^(?:the|said|of|in)\s+", "", regex=True)
                     .str.replace(r"^saint\s+", "st ", regex=True)
                     .str.replace(r"[^a-z ]", "", regex=True).str.strip())
cty = pl[pl.county.notna()]
m = r.merge(cty[["state", "county_key", "place_id"]], on=["state", "county_key"], how="left")
uniq = (cty.groupby("county_key").filter(lambda g: len(g) == 1)[["county_key", "place_id", "state"]]
           .rename(columns={"place_id": "pid_u", "state": "state_u"}))
m = m.merge(uniq, on="county_key", how="left")
ns = m.state.isna() & m.pid_u.notna()
m.loc[ns, "place_id"] = m.loc[ns, "pid_u"]; m.loc[ns, "state"] = m.loc[ns, "state_u"]
m["precision"] = pd.Series("county", index=m.index).where(m.place_id.notna())
so = m.place_id.isna() & m.state.isin(pl.place_id)
m.loc[so, "place_id"] = m.loc[so, "state"]; m.loc[so, "precision"] = "state"
print("resolved", m.place_id.notna().sum(), "of", len(m), m.precision.value_counts())

# 5.5
res = m[m.place_id.notna()].copy()
res["date"] = res.year.where(res.kind != "now_resides", res.app_year)
res["snippet"] = res.window.str[:100]
res = res.drop_duplicates(["NAID", "person", "kind", "place_id"])
has_cty = res[res.precision == "county"][["NAID", "person", "kind", "state"]].drop_duplicates().assign(_c=True)
res = res.merge(has_cty, on=["NAID", "person", "kind", "state"], how="left")
res = res[~((res.precision == "state") & res._c.fillna(False))].drop(columns="_c")
nr = res.kind == "now_resides"
best = (res[nr].assign(p=res.precision.map({"county": 0, "state": 1}))
          .sort_values(["NAID", "person", "date", "p"], ascending=[True, True, False, True], na_position="last")
          .drop_duplicates(["NAID", "person"]).drop(columns="p"))
res = pd.concat([res[~nr], best])
ORDER = {"born": 0, "enlisted": 1, "lived_since": 2, "removed_to": 2, "married": 2, "now_resides": 3}
res = res.assign(k=res.kind.map(ORDER)).sort_values(["NAID", "person", "k", "date", "pageNumber"], na_position="last")
res["seq"] = res.groupby(["NAID", "person"]).cumcount() + 1
res["method"] = "declaration_regex"
COLS = ["NAID", "seq", "person", "place_id", "state", "precision", "kind", "date", "pageNumber", "snippet", "method"]
res = res[COLS]
print(len(res), res.NAID.nunique())
print(pd.crosstab(res.kind, res.precision))

# 5.6
sm = io.read_stage("interim", "state_mentions")
sm["state"] = places.normalise_state_series(sm.state_raw)
sm = sm[sm.state.notna() & (sm.state != "Cont")].groupby(["NAID", "state"], as_index=False).n.sum()
freq = sm.sort_values(["NAID", "n", "state"], ascending=[True, False, True]).drop_duplicates("NAID")
fb = freq[~freq.NAID.isin(res.loc[res.kind == "now_resides", "NAID"])]
fb = fb.assign(seq=99, person="unknown", place_id=fb.state, precision="state", kind="now_resides",
               date=pd.NA, pageNumber=pd.NA, snippet="n=" + fb.n.astype(str), method="most_frequent")
res = pd.concat([res, fb[COLS]], ignore_index=True)

# 5.7
assert res.place_id.isin(pl.place_id).all()
assert not res.duplicated(["NAID", "person", "kind", "place_id"]).any()
assert res.groupby(["NAID", "person"]).kind.apply(lambda k: (k == "now_resides").sum() <= 1).all()
io.write_stage(res, "interim", "residences")


kind
now_resides    132413
born            23200
removed_to      18532
married         16692
enlisted         9828
lived_since      1108
Name: count, dtype: int64
resolved 148547 of 201773 precision
state     90343
county    58204
Name: count, dtype: int64
101967 56402
precision    county  state
kind                      
born           6471   7765
enlisted       2059   2714
lived_since      84    189
married        2483   3708
now_resides   31335  34226
removed_to     4714   6219


PosixPath('/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/data/interim/residences.parquet')